# 02 – Preprocessing & Data Cleaning

### Purpose of the Notebook
This notebook applies systematic cleaning and standardisation to the pre‑saved datasets (dataset.pkl and dataset_de.pkl).
All decisions are based on the insights from Notebook 01_data_overview (EDA), including handling of missing data, removal of low‑quality fields, type corrections, logical consistency checks, and creation of derived features.

### Steps
- Load pre‑saved datasets
- Apply preprocessing pypline
- Save cleaned datasets

--------------------
#### Imports & Setup & Dataset
-------------------

In [1]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [26]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.preprocessing import preprocess
from my_scripts.eda import overview

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset.pkl")

print("EU dataset:", df.shape)

EU dataset: (4362869, 75)


--------------
### Apply preprocessing pipeline
-----------

In [27]:
# apply funktion
df_clean = preprocess(df)


In [28]:
# shape of the cleaned dataset
print(df_clean.shape)

(3003042, 12)


In [29]:
# inspekt of the cleaned dataset
overview(df_clean)

,dtype,total,missing_n,missing_%,uniques_n,uniques
YEAR,int16,3003042,0,0.00,9,"[2008, 2009, 2010, 2011, 2012, 2013, 2014, 201..."
ISO_COUNTRY_CODE,str,3003042,0,0.00,33,"[DE, ES, FR, PL, HU, IT, CY, UK, RO, PT, SE, N..."
CAE_TYPE,str,3003042,0,0.00,10,"[8, 3, 1, 6, R, N, 4, 5A, 5, Z]"
TYPE_OF_CONTRACT,str,3003042,0,0.00,3,"[W, U, S]"
TOP_TYPE,str,3003042,0,0.00,10,"[OPE, NIC, RES, NOC, Unknown, COD, AWP, NOP, N..."
MAIN_CPV_CODE_GPA,str,3003042,0,0.00,68,"[Unknown, 38.0, 75.0, 14.0, 30.0, 794.0, 73.0,..."
AWARD_VALUE_EURO,float64,3003042,0,0.00,1603450,"[302964.15, 478780.08, 14237368.1, 5204457.56,..."
CRIT_PRICE_WEIGHT,float64,1472948,1530094,50.95,167,"[nan, 100.0, 70.0, 95.0, 85.0, 60.0, 45.0, 50...."
NUMBER_OFFERS,float64,3003042,0,0.00,363,"[2.0, 3.0, 6.0, 9.0, 45.0, 1.0, 5.0, 4.0, 10.0..."
DT_DISPATCH,datetime64[us],3003042,0,0.00,3273,"[2007-12-11 00:00:00, 2007-12-27 00:00:00, 200..."


In [30]:
df_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
YEAR,"3,003,042.00","2,012.35","2,008.00","2,010.00","2,012.00","2,015.00","2,016.00",2.53
AWARD_VALUE_EURO,"3,003,042.00","6,228,494.44",0.00,"6,239.02","63,369.53","411,014.43","9,999,999,999.00","91,360,294.19"
CRIT_PRICE_WEIGHT,"1,472,948.00",100.34,0.00,100.00,100.00,100.00,"1,487,600.00","1,287.99"
NUMBER_OFFERS,"3,003,042.00",7.05,0.00,1.00,3.00,6.00,500.00,18.65
DT_DISPATCH,3003042,2012-11-02 06:26:55.553827,2007-07-16 00:00:00,2010-10-01 00:00:00,2012-12-18 00:00:00,2015-01-21 00:00:00,2016-12-30 00:00:00,NaN
DT_AWARD,2966794,2012-09-14 06:34:24.177291,1996-09-25 00:00:00,2010-08-06 00:00:00,2012-11-02 00:00:00,2014-12-09 00:00:00,2028-10-04 00:00:00,NaN


#### Notes: Summary of Data Cleaning Results

1. Dataset Size After Preprocessing
- Before: 4,039,906 rows × 75 columns
- After: 3,003,042 rows × 12 columns
The reduction reflects removal of non‑informative fields, consolidation of text columns, elimination of corrupted numeric values, and reconstruction of award amounts from multiple financial fields.

2. Columns Removed During Preprocessing
Removed due to >40% missing values or structural corruption
- VALUE_EURO_FIN_1, VALUE_EURO_FIN_2
- AWARD_VALUE_EURO_FIN_1
- AWARD_EST_VALUE_EURO
- Winner information (WIN_*)
- Contracting authority details (CAE_*)
- GPA-related fields
- Award criteria weights (CRIT_*, except CRIT_PRICE_WEIGHT)
- Additional CPVs
- Procedural flags with extreme cardinality (B_MULTIPLE_, B_FRA_, FRA_ESTIMATED, etc.)
- TED_NOTICE_URL
Removed due to irrelevance for competition modelling
- Identifiers (ID_NOTICE_CAN, ID_AWARD, ID_LOT_AWARDED, CONTRACT_NUMBER)
- Non-award information (INFO_ON_NON_AWARD, INFO_UNPUBLISHED)
- Administrative metadata (MAIN_ACTIVITY, EU_INST_CODE)
Removed due to consolidation
- TITLE - merged into a single NLP field used for TF‑IDF/SVD/NMF/SVM.

3. Reconstruction and Cleaning of Award Values
3.1. Multi-source reconstruction of AWARD_VALUE_EURO
Missing award values were filled using any realistic numeric value (≤10 billion EUR) found in:
- VALUE_EURO
- VALUE_EURO_FIN_1
- VALUE_EURO_FIN_2
- AWARD_EST_VALUE_EURO
- AWARD_VALUE_EURO_FIN_1
This step reduced missing award values from ~60% to <15%.

3.2. Removal of unrealistic values
All values above 10,000,000,000 EUR were removed as technical artifacts.

3.3. Final removal of remaining missing award values
After reconstruction, remaining missing AWARD_VALUE_EURO (~15%) were dropped, as tenders without any valid financial information cannot be used for competition or failure‑risk modelling.

4. Additional Preprocessing Steps
- CANCELLED tenders removed entirely.
- All categorical missing values replaced with "Unknown".
- All categorical variables converted to string to ensure stable feature engineering.
- Numeric columns converted to float with coercion to handle corrupted formats.
- Date fields parsed into datetime.
- High-cardinality categorical fields removed to prevent memory issues and unstable encoding.

--------------
### Save cleaned datasets

-----------

In [31]:
df_clean.to_pickle("../data/dataset_clean.pkl")